# Explore UBL OASIS Google Drive Folder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/setup-ubl-sheets-access-zWGFj/notebooks/explore-drive-folder.ipynb)

Recursively explores the OASIS UBL TC Google Drive folder and discovers
**all** files and their revision history. Saves results to Google Drive
so they can be analyzed later.

## What this notebook does

1. **Lists every file** in the OASIS UBL TC shared folder (recursively)
2. **Fetches revision history** for every Google Sheets spreadsheet found
3. **Identifies which files have explorable revision history** beyond the
   already-known UBL 2.5 Library and Documents sheets
4. **Saves a complete inventory** as JSON to Google Drive

## Known sheets (already explored)

| Sheet | ID | Revisions | Status |
|-------|----|-----------|--------|
| UBL 2.5 Library | `18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY` | ~2005 | Fully archived |
| UBL 2.5 Documents | `1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg` | ~2204 | Fully archived |
| UBL 2.5 Signature | `1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g` | ? | Not yet explored |
| UBL 2.4 Library | `1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs` | ? | Not yet explored |
| UBL 2.4 Documents | `1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y` | ? | Not yet explored |

## Output

```
Drive: ubl-gc-revisions/
├── drive-discovery.json              ← complete folder tree + revision counts
├── drive-discovery-summary.txt       ← human-readable summary
└── (existing revision archives...)
```

In [ ]:
# === Step 0: Auth + Mount Drive ===
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import google.auth
from google.auth.transport.requests import Request as AuthRequest
from datetime import datetime, timezone

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive.readonly']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')
if creds.expiry:
    remaining = (creds.expiry - datetime.now(timezone.utc).replace(tzinfo=None)).total_seconds()
    print(f'Token expiry: {creds.expiry.isoformat()} ({remaining:.0f}s from now)')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

## Step 1: Configuration & Helpers

In [ ]:
import json, time, sys
from urllib.request import Request, urlopen
from urllib.error import HTTPError

# Root folder: OASIS UBL TC shared Google Drive folder
ROOT_FOLDER_ID = '0B4X4evii3UjcdG5wNlVFTXlaYVU'

# Already-explored sheet IDs (for tagging in output)
KNOWN_SHEETS = {
    '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY': 'ubl25_library (ARCHIVED)',
    '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg': 'ubl25_documents (ARCHIVED)',
    '1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g': 'ubl25_signature (KNOWN)',
    '1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs': 'ubl24_library (KNOWN)',
    '1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y': 'ubl24_documents (KNOWN)',
}

# Rate limiting
API_DELAY = 0.3  # seconds between API calls

# Provenance tracking
api_log = []


def now_iso():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')


def ensure_fresh_token():
    """Refresh TOKEN if near expiry."""
    global TOKEN
    if not creds.expiry:
        return
    remaining = (creds.expiry - datetime.now(timezone.utc).replace(tzinfo=None)).total_seconds()
    if remaining > 300:
        return
    old = TOKEN[:8]
    creds.token = None
    creds.refresh(AuthRequest())
    TOKEN = creds.token
    print(f'  >> Token refreshed: {old}... -> {TOKEN[:8]}...')


def api_get(url, context=''):
    """Authenticated GET with retry, rate limiting, and provenance logging."""
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        call_time = now_iso()
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': 200, 'attempt': attempt + 1, 'context': context,
            })
            time.sleep(API_DELAY)
            return data
        except HTTPError as e:
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': e.code, 'attempt': attempt + 1, 'context': context,
            })
            if e.code in (429, 500, 502, 503) and attempt < 3:
                wait = 2 ** (attempt + 1)
                print(f'    [{e.code}] retrying in {wait}s...')
                time.sleep(wait)
                continue
            print(f'    ERROR {e.code}: {context}')
            return None
        except Exception as exc:
            api_log.append({
                'timestamp': call_time, 'url': url,
                'status': 0, 'attempt': attempt + 1, 'context': context,
                'error': str(exc),
            })
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue
            print(f'    EXCEPTION: {exc}')
            return None
    return None


print('Helpers ready')

## Step 2: Recursive Folder Exploration

Walks the entire OASIS UBL TC Google Drive folder tree. For every
Google Sheets spreadsheet found, fetches its complete revision history
via the Drive API v3 Revisions endpoint.

In [ ]:
def list_folder(folder_id):
    """List all files in a folder (paginated)."""
    all_files = []
    page_token = None
    page = 0
    while True:
        page += 1
        url = (
            f'https://www.googleapis.com/drive/v3/files'
            f'?q=%27{folder_id}%27+in+parents+and+trashed%3Dfalse'
            f'&fields=nextPageToken,files(id,name,mimeType,modifiedTime,'
            f'createdTime,size,owners/displayName,owners/emailAddress,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
            f'shared,webViewLink)'
            f'&pageSize=100&orderBy=name'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        data = api_get(url, f'list:{folder_id}:p{page}')
        if not data:
            break
        all_files.extend(data.get('files', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
    return all_files


def get_revisions(file_id, file_name):
    """Fetch all revisions of a file (paginated)."""
    all_revs = []
    page_token = None
    page = 0
    while True:
        page += 1
        url = (
            f'https://www.googleapis.com/drive/v3/files/{file_id}/revisions'
            f'?pageSize=1000'
            f'&fields=nextPageToken,revisions(id,modifiedTime,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
            f'size,exportLinks)'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        data = api_get(url, f'revisions:{file_id}:{file_name}:p{page}')
        if not data:
            break
        all_revs.extend(data.get('revisions', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
    return all_revs


def explore_folder(folder_id, folder_name, path='/', depth=0):
    """Recursively explore a folder and fetch revision history for spreadsheets."""
    indent = '  ' * depth
    print(f'{indent}[folder] {folder_name}/')
    
    files = list_folder(folder_id)
    print(f'{indent}  {len(files)} items')
    
    result = {
        'id': folder_id,
        'name': folder_name,
        'path': path,
        'type': 'folder',
        'children': [],
    }
    
    for f in files:
        mime = f.get('mimeType', '')
        name = f.get('name', '?')
        fid = f.get('id', '')
        child_path = f'{path}{name}'
        
        entry = {
            'id': fid,
            'name': name,
            'path': child_path,
            'mimeType': mime,
            'modifiedTime': f.get('modifiedTime'),
            'createdTime': f.get('createdTime'),
            'size': f.get('size'),
            'owners': f.get('owners'),
            'lastModifyingUser': f.get('lastModifyingUser'),
            'webViewLink': f.get('webViewLink'),
        }
        
        if mime == 'application/vnd.google-apps.folder':
            sub = explore_folder(fid, name, f'{child_path}/', depth + 1)
            entry['children'] = sub['children']
            entry['type'] = 'folder'
        
        elif mime == 'application/vnd.google-apps.spreadsheet':
            entry['type'] = 'spreadsheet'
            known_tag = KNOWN_SHEETS.get(fid)
            if known_tag:
                entry['known_as'] = known_tag
            
            revisions = get_revisions(fid, name)
            entry['revision_count'] = len(revisions)
            entry['revisions'] = revisions
            
            if revisions:
                entry['first_revision_time'] = revisions[0].get('modifiedTime')
                entry['last_revision_time'] = revisions[-1].get('modifiedTime')
                authors = set()
                for rev in revisions:
                    user = rev.get('lastModifyingUser', {})
                    dn = user.get('displayName') or user.get('emailAddress')
                    if dn:
                        authors.add(dn)
                entry['unique_authors'] = sorted(authors)
            
            tag = f' [{known_tag}]' if known_tag else ''
            print(f'{indent}  [sheet] {name}: {len(revisions)} revisions{tag}')
        
        elif mime == 'application/vnd.google-apps.shortcut':
            entry['type'] = 'shortcut'
            target_url = (
                f'https://www.googleapis.com/drive/v3/files/{fid}'
                f'?fields=shortcutDetails(targetId,targetMimeType,targetResourceKey)'
            )
            target_data = api_get(target_url, f'shortcut:{fid}:{name}')
            if target_data and 'shortcutDetails' in target_data:
                entry['shortcutTarget'] = target_data['shortcutDetails']
                tid = target_data['shortcutDetails'].get('targetId', '?')
                tmime = target_data['shortcutDetails'].get('targetMimeType', '?')
                print(f'{indent}  [shortcut] {name} -> {tid[:12]}... ({tmime})')
                
                # Follow shortcuts to spreadsheets
                if tmime == 'application/vnd.google-apps.spreadsheet':
                    revisions = get_revisions(tid, f'(shortcut) {name}')
                    entry['shortcutTarget']['revision_count'] = len(revisions)
                    entry['shortcutTarget']['revisions'] = revisions
                    if revisions:
                        entry['shortcutTarget']['first_revision_time'] = revisions[0].get('modifiedTime')
                        entry['shortcutTarget']['last_revision_time'] = revisions[-1].get('modifiedTime')
                    known_tag = KNOWN_SHEETS.get(tid)
                    if known_tag:
                        entry['shortcutTarget']['known_as'] = known_tag
                    tag = f' [{known_tag}]' if known_tag else ''
                    print(f'{indent}    -> {len(revisions)} revisions{tag}')
        
        else:
            entry['type'] = 'file'
            size_str = f' ({int(f.get("size", 0)):,}b)' if f.get('size') else ''
            print(f'{indent}  [file] {name}{size_str}')
        
        result['children'].append(entry)
    
    return result


print('Exploration functions ready')

In [ ]:
# === Run the exploration ===
start_time = now_iso()
print(f'Started: {start_time}')
print(f'Root folder: {ROOT_FOLDER_ID}')
print(f'URL: https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}')
print()

tree = explore_folder(ROOT_FOLDER_ID, 'UBL TC (root)')

end_time = now_iso()
print(f'\nCompleted: {end_time}')
print(f'API calls: {len(api_log)}')

## Step 3: Analyze Results

Extract all spreadsheets, count their revisions, and identify which
ones have significant revision history that hasn't been explored yet.

In [ ]:
def collect_spreadsheets(node, results=None):
    """Recursively collect all spreadsheet entries from the tree."""
    if results is None:
        results = []
    
    for child in node.get('children', []):
        t = child.get('type')
        if t == 'spreadsheet':
            results.append(child)
        elif t == 'shortcut':
            target = child.get('shortcutTarget', {})
            if target.get('revision_count', 0) > 0:
                # Create a synthetic entry for the shortcut target
                results.append({
                    'id': target.get('targetId'),
                    'name': f"(shortcut) {child['name']}",
                    'path': child.get('path', ''),
                    'type': 'spreadsheet',
                    'revision_count': target.get('revision_count', 0),
                    'first_revision_time': target.get('first_revision_time'),
                    'last_revision_time': target.get('last_revision_time'),
                    'known_as': target.get('known_as'),
                    'via_shortcut': True,
                    'revisions': target.get('revisions', []),
                })
        elif t == 'folder':
            collect_spreadsheets(child, results)
    
    return results


def count_all(node):
    """Count totals in the tree."""
    folders = files = sheets = shortcuts = revisions = 0
    for child in node.get('children', []):
        t = child.get('type')
        if t == 'folder':
            folders += 1
            f2, fi2, s2, sc2, r2 = count_all(child)
            folders += f2; files += fi2; sheets += s2
            shortcuts += sc2; revisions += r2
        elif t == 'spreadsheet':
            sheets += 1
            revisions += child.get('revision_count', 0)
        elif t == 'shortcut':
            shortcuts += 1
            revisions += child.get('shortcutTarget', {}).get('revision_count', 0)
        else:
            files += 1
    return folders, files, sheets, shortcuts, revisions


# Collect all spreadsheets
all_sheets = collect_spreadsheets(tree)
folders, files, sheets, shortcuts, total_revs = count_all(tree)

print('=' * 70)
print('FOLDER INVENTORY')
print('=' * 70)
print(f'  Folders:       {folders}')
print(f'  Files:         {files}')
print(f'  Spreadsheets:  {sheets}')
print(f'  Shortcuts:     {shortcuts}')
print(f'  Total revisions (Drive API): {total_revs}')
print()

# Sort by revision count (most revisions first)
all_sheets.sort(key=lambda s: s.get('revision_count', 0), reverse=True)

print('=' * 70)
print('ALL SPREADSHEETS (by revision count)')
print('=' * 70)
print()

already_archived_ids = {
    '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

new_discoveries = []

for i, s in enumerate(all_sheets, 1):
    fid = s.get('id', '?')
    name = s.get('name', '?')
    rc = s.get('revision_count', 0)
    first = (s.get('first_revision_time') or '?')[:10]
    last = (s.get('last_revision_time') or '?')[:10]
    known = s.get('known_as', '')
    path = s.get('path', '')
    
    status = ''
    if fid in already_archived_ids:
        status = ' [ARCHIVED]'
    elif known:
        status = f' [{known}]'
    elif rc > 1:
        status = ' *** NEW ***'
        new_discoveries.append(s)
    
    shortcut_tag = ' (via shortcut)' if s.get('via_shortcut') else ''
    
    print(f'{i:3d}. {name}')
    print(f'     ID: {fid}')
    print(f'     Path: {path}')
    print(f'     Revisions: {rc} ({first} -> {last}){status}{shortcut_tag}')
    
    # Show authors for sheets with significant history
    if rc > 5:
        authors = set()
        for rev in s.get('revisions', []):
            user = rev.get('lastModifyingUser', {})
            dn = user.get('displayName') or user.get('emailAddress')
            if dn:
                authors.add(dn)
        if authors:
            print(f'     Authors: {", ".join(sorted(authors))}')
    print()

print('=' * 70)
print(f'NEW DISCOVERIES (sheets with >1 revision, not already known)')
print('=' * 70)

if new_discoveries:
    for s in new_discoveries:
        rc = s.get('revision_count', 0)
        name = s.get('name', '?')
        fid = s.get('id', '?')
        first = (s.get('first_revision_time') or '?')[:10]
        last = (s.get('last_revision_time') or '?')[:10]
        print(f'  {name}')
        print(f'    ID: {fid}, {rc} revisions ({first} -> {last})')
else:
    print('  (none found)')

print()

## Step 4: Detailed Revision Timeline

For each spreadsheet with significant revision history (>5 revisions),
show a timeline of edits with timestamps and authors.

In [ ]:
# Show detailed revision timeline for sheets with significant history
DETAIL_THRESHOLD = 5  # show details for sheets with more than this many revisions

significant_sheets = [
    s for s in all_sheets
    if s.get('revision_count', 0) > DETAIL_THRESHOLD
]

print(f'Sheets with >{DETAIL_THRESHOLD} revisions: {len(significant_sheets)}')
print()

for s in significant_sheets:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    revisions = s.get('revisions', [])
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    
    print(f'={"="*68}')
    print(f'{name}{tag}')
    print(f'  ID: {fid}')
    print(f'  Revisions: {len(revisions)}')
    print(f'={"="*68}')
    
    # Show first 10, last 10, and any large gaps
    if len(revisions) <= 30:
        show_revs = revisions
    else:
        show_revs = revisions[:10]
        show_revs.append({'_separator': True, '_count': len(revisions) - 20})
        show_revs.extend(revisions[-10:])
    
    for j, rev in enumerate(show_revs):
        if rev.get('_separator'):
            print(f'  ... ({rev["_count"]} more revisions) ...')
            continue
        
        rev_id = rev.get('id', '?')
        mod_time = rev.get('modifiedTime', '?')
        user = rev.get('lastModifyingUser', {})
        author = user.get('displayName') or user.get('emailAddress', '?')
        size = rev.get('size', '?')
        
        print(f'  rev {rev_id:>5s}  {mod_time[:19]}  {author:30s}  {size}')
    
    # Activity analysis: edits per month
    print(f'\n  Monthly activity:')
    months = {}
    for rev in revisions:
        mt = rev.get('modifiedTime', '')
        if mt and len(mt) >= 7:
            month = mt[:7]
            months[month] = months.get(month, 0) + 1
    
    for month in sorted(months.keys()):
        bar = '#' * min(months[month], 60)
        print(f'    {month}: {months[month]:4d} {bar}')
    
    print()

## Step 5: Internal Revision Count Probe

The Drive API `revisions.list` returns a limited set of revisions
(typically 25-100). The actual **internal revision counter** can be much
higher (the UBL 2.5 Library sheet has 2005+ internal revisions despite
the API listing only ~25).

For each non-archived spreadsheet with significant history, probe for
the true max revision number by testing export URLs with binary search.

In [ ]:
def probe_max_revision(sheet_id, start_guess=100, max_probe=10000):
    """Binary search for the highest valid internal revision number.
    
    Google Sheets' internal revision counter is independent of the
    Drive API revision list. The export URL accepts any valid revision
    number and returns 400/404 for invalid ones.
    
    Returns (max_rev, probes_used).
    """
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    probes = 0
    
    def test_rev(n):
        nonlocal probes
        probes += 1
        url = (
            f'https://docs.google.com/spreadsheets/export'
            f'?id={sheet_id}&revision={n}&exportFormat=csv'
        )
        try:
            req = Request(url, headers=headers, method='HEAD')
            with urlopen(req, timeout=15) as resp:
                return resp.status == 200
        except HTTPError:
            return False
        except Exception:
            return False
    
    # Phase 1: Find upper bound by doubling
    upper = start_guess
    while upper <= max_probe:
        if test_rev(upper):
            upper *= 2
            time.sleep(0.5)
        else:
            break
        time.sleep(0.5)
    
    if upper > max_probe:
        return max_probe, probes  # hit ceiling
    
    # Phase 2: Binary search between upper/2 and upper
    lo = max(1, upper // 2)
    hi = upper
    
    while lo < hi - 1:
        mid = (lo + hi) // 2
        if test_rev(mid):
            lo = mid
        else:
            hi = mid
        time.sleep(0.5)
    
    return lo, probes


# Probe non-archived sheets
sheets_to_probe = [
    s for s in all_sheets
    if s.get('revision_count', 0) > 0
    and s.get('id') not in already_archived_ids
    and s.get('type') == 'spreadsheet'
]

print(f'Probing {len(sheets_to_probe)} sheets for true max revision...\n')

probe_results = []

for s in sheets_to_probe:
    fid = s.get('id', '?')
    name = s.get('name', '?')
    api_rev_count = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    
    print(f'  {name}{tag} (API says {api_rev_count} revisions)...', end=' ', flush=True)
    
    max_rev, probes = probe_max_revision(fid)
    ratio = max_rev / api_rev_count if api_rev_count > 0 else 0
    
    result = {
        'id': fid,
        'name': name,
        'api_revision_count': api_rev_count,
        'internal_max_revision': max_rev,
        'ratio': round(ratio, 1),
        'probes_used': probes,
        'known_as': known,
    }
    probe_results.append(result)
    
    interesting = ' *** INTERESTING ***' if max_rev > 50 else ''
    print(f'max_rev={max_rev} (ratio={ratio:.1f}x, {probes} probes){interesting}')
    time.sleep(1)  # be gentle

# Summary
print(f'\n{"="*70}')
print(f'INTERNAL REVISION PROBE RESULTS')
print(f'{"="*70}')

probe_results.sort(key=lambda r: r['internal_max_revision'], reverse=True)

for r in probe_results:
    tag = f' [{r["known_as"]}]' if r['known_as'] else ''
    flag = ' <<<' if r['internal_max_revision'] > 50 else ''
    print(f'  {r["name"]:45s} API: {r["api_revision_count"]:5d}  '
          f'Internal: {r["internal_max_revision"]:6d}  '
          f'Ratio: {r["ratio"]:6.1f}x{tag}{flag}')

## Step 6: Save Results to Drive

In [ ]:
# Build the output
output = {
    '_provenance': {
        'description': 'Complete inventory of OASIS UBL TC Google Drive shared folder',
        'root_folder_url': f'https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}',
        'root_folder_id': ROOT_FOLDER_ID,
        'started_at': start_time,
        'completed_at': end_time,
        'total_api_calls': len(api_log),
    },
    'stats': {
        'total_folders': folders,
        'total_files': files,
        'total_spreadsheets': sheets,
        'total_shortcuts': shortcuts,
        'total_api_revisions': total_revs,
    },
    'spreadsheet_summary': [
        {
            'name': s.get('name'),
            'id': s.get('id'),
            'path': s.get('path'),
            'revision_count': s.get('revision_count', 0),
            'first_revision': s.get('first_revision_time'),
            'last_revision': s.get('last_revision_time'),
            'known_as': s.get('known_as'),
            'via_shortcut': s.get('via_shortcut', False),
        }
        for s in all_sheets
    ],
    'probe_results': probe_results,
    'tree': tree,
    'api_log': api_log,
}

# Save to Drive
discovery_path = DRIVE_DIR / 'drive-discovery.json'
discovery_path.write_text(json.dumps(output, indent=2))
print(f'Saved: {discovery_path} ({discovery_path.stat().st_size:,} bytes)')

# Human-readable summary
summary_lines = []
summary_lines.append('OASIS UBL TC Google Drive - Complete Inventory')
summary_lines.append(f'Generated: {end_time}')
summary_lines.append(f'Root: https://drive.google.com/drive/folders/{ROOT_FOLDER_ID}')
summary_lines.append('')
summary_lines.append(f'Folders: {folders}, Files: {files}, Spreadsheets: {sheets}, Shortcuts: {shortcuts}')
summary_lines.append(f'Total API-listed revisions: {total_revs}')
summary_lines.append('')
summary_lines.append('SPREADSHEETS BY REVISION COUNT:')
summary_lines.append('')

for i, s in enumerate(all_sheets, 1):
    fid = s.get('id', '?')
    name = s.get('name', '?')
    rc = s.get('revision_count', 0)
    known = s.get('known_as', '')
    tag = f' [{known}]' if known else ''
    first = (s.get('first_revision_time') or '?')[:10]
    last = (s.get('last_revision_time') or '?')[:10]
    summary_lines.append(f'{i:3d}. {name}: {rc} revisions ({first} -> {last}){tag}')
    summary_lines.append(f'     ID: {fid}')

if probe_results:
    summary_lines.append('')
    summary_lines.append('INTERNAL REVISION PROBES:')
    summary_lines.append('')
    for r in probe_results:
        tag = f' [{r["known_as"]}]' if r['known_as'] else ''
        summary_lines.append(
            f'  {r["name"]}: API={r["api_revision_count"]}, '
            f'Internal max={r["internal_max_revision"]}, '
            f'Ratio={r["ratio"]}x{tag}'
        )

if new_discoveries:
    summary_lines.append('')
    summary_lines.append('NEW DISCOVERIES (not previously known):')
    summary_lines.append('')
    for s in new_discoveries:
        summary_lines.append(f'  {s["name"]}: {s.get("revision_count", 0)} revisions')
        summary_lines.append(f'    ID: {s["id"]}')

summary_text = '\n'.join(summary_lines)
summary_path = DRIVE_DIR / 'drive-discovery-summary.txt'
summary_path.write_text(summary_text)
print(f'Saved: {summary_path} ({summary_path.stat().st_size:,} bytes)')

print(f'\n--- Summary ---\n')
print(summary_text)

## Step 7: Next Steps

After running this notebook:

1. **Review the summary** above to see which sheets have revision history
2. **Check the probe results** — sheets with high internal revision counts
   (>50) are candidates for full revision archival using the
   `download-all-revisions.ipynb` notebook
3. **Copy the `drive-discovery.json`** back to the repo for analysis:
   ```
   Drive: ubl-gc-revisions/drive-discovery.json
   ```

### What to look for

- **UBL 2.4 Library/Documents**: These are the most likely candidates
  for significant revision history. If the internal revision count is
  high (>100), they could contain the complete 2.4 development history.

- **Signature sheet**: This sheet is small and rarely changes, so it
  likely has few meaningful revisions.

- **Unknown sheets**: Any spreadsheet we didn't know about that has
  significant revision history could be a valuable discovery.